# Tokenizer

In this section, we are going to build a simple tokenizer that allows us to tokenize our sentences into smaller components called tokens and decode input tokens into the original string. Each token has its own unique identifier.

In the tokenizer, we have two main processes:
- **Encoding**: Chunking our sentences into smaller components called token. Each token will be represented with a unique identifier
<br>
<img height="400px" width="700px" src="images/tokenizer_encoding.png"/>
- **Decoder**: Converting token ids into the original sentence

In this notebook, we will build a tokenizer using an algorithm called **Byte Pair Encoding**, which is used in building many large language models such as ChatGPT or Gemini.


In [2]:
import re
# Start with dummy example
sentence = "Hello, my name is Tim. Nice to meet you"

tokens = re.split(r"\s", sentence)

token_ids = []
token_to_id = {}
id_to_token = {}
i = 0
for token in tokens:
    if token_to_id.get(token, None) is None:
        token_to_id[token] = i
        id_to_token[i] = token
        i+=1
    
print(token_to_id)




{'Hello,': 0, 'my': 1, 'name': 2, 'is': 3, 'Tim.': 4, 'Nice': 5, 'to': 6, 'meet': 7, 'you': 8}


In [3]:
sentence_ids = [1, 4, 0, 8, 3, 2]
decoded_sentence = ""

for id in sentence_ids:
    decoded_sentence += id_to_token.get(id, "") + " "
print("Decoded sentence: ", decoded_sentence)

Decoded sentence:  my Tim. Hello, you is name 


## Importing dataset

Now let's build our tokenizer vocabulary by training it with more text data

In [46]:
# Import book text dataset 
with open("dataset/how_we_think.txt", 'r', encoding='utf-8-sig') as f:
    lines = f.readlines()
    # Merge those lines together
    text = "".join(lines)
    

### Byte Pair Encoding

#### Materials
- [Byte Pair Encoding Hugging Face](https://www.youtube.com/watch?v=HEikzVL-lZU)

#### Core ideas

**Rule 1**: Do not split frequently used words into smaller subwords

**Rule 2**: Split the rare words into smaller, meaningful subwords

- Eg: "play" should not be split. "playing" should be split into "play" and "ing"

#### Benefits
1. "plays" and "playing" comes from the same root "play"
2. Some words have different root words but share the suffix part such as "classification" and "location" which both share "cation"

**BFE algorithm**: Most common pair of consecutive bytes of data is replaced with a byte that does not occur in the data

In [102]:
# Build a Tokenizer Class
class Tokenizer:
    def __init__(self):
        self.token_to_id = {'<start>' : 1, '<end>' : 2, ' ' : 3, '<unknown>' : 4}
        self.id_to_token = {1 : '<start>', 2 : '<end>', 3 : ' ', 4: "<unknown>"}
        self.vocab = set(['<start>', '<end>', '<unknown>', ' '])
        self.id_counter = 5
    def train(self, text):
        assert len(text) > 0, "You must input a non-empty text"
        # Replace space with special token \w
        words = text.split(" ")
        text_characters = []
        for word in words:
            for char in word:
                if char not in self.vocab:
                    self.token_to_id[char] = self.id_counter
                    self.id_to_token[self.id_counter] = char
                    self.vocab.add(char)
                    self.id_counter += 1
                text_characters.append(char)

        stop = False
        while not stop:
            frequency = {}
            # Contruct frequency from the text_characters
            for i in range(1, len(text_characters)):
                pair = text_characters[i-1] + text_characters[i]
                if pair not in frequency:
                    frequency[pair] = 0
                frequency[pair] += 1

            most_commmon_pair, occurence = max(frequency.items(), key = lambda item: item[1])
            if occurence > 1:
                self.token_to_id[most_commmon_pair] = self.id_counter
                self.id_to_token[self.id_counter] = most_commmon_pair
                self.id_counter += 1
                self.vocab.add(most_commmon_pair)

                # Merge those tokens inside the text_characters
                new_text_characters = []

                index = 0
                while (index < len(text_characters) - 1):
                    pair = text_characters[index] + text_characters[index+1]
                    if (pair == most_commmon_pair):
                        new_text_characters.append(pair)
                        index += 2 # Skip the next character
                    else:
                        new_text_characters.append(text_characters[index])
                        index += 1
                text_characters = new_text_characters
            else:
                stop = True

    def encode(self, inputs: str):
        tokens = [inputs[0]]
        for char in inputs[1:]:
            new_char = tokens[-1] + char
            if new_char in self.vocab:
                tokens[-1] = new_char
            else:
                tokens.append(char)

        # Convert into ids
        ids = [self.token_to_id['<start>']] + [self.token_to_id.get(token, self.token_to_id['<unknown>']) for token in tokens] + [self.token_to_id['<end>']]
        return ids

    def decode(self, inputs):
        string =  "".join([self.id_to_token.get(id, self.id_to_token[4]) for id in inputs])
        return string

In [93]:
a = [""]
a[-1]

''

In [103]:
tokenizer = Tokenizer()

In [ ]:
# Train tokenizer
tokenizer.train(text)

1

In [104]:
import pickle
with open('id_to_token.pkl', 'rb') as f:
    id_to_token = pickle.load(f)
    id_to_token[3] = ' '
    tokenizer.id_to_token = id_to_token

In [105]:
with open("token_to_id.pkl", "rb") as f:
    token_to_id = pickle.load(f)
    token_to_id.pop("<\w>")
    token_to_id[" "] = 3
    tokenizer.token_to_id = token_to_id
    

In [106]:
sentence = "My name is Khai"

tokenizer.encode(sentence)

[1, 59, 29, 3, 16, 28, 33, 7, 3, 25, 27, 3, 66, 6, 28, 25, 2]

In [107]:
ids = [1, 59, 29, 3, 16, 28, 33, 7, 3, 25, 27, 3, 66, 6, 28, 25, 2]

tokenizer.decode(ids)

'<start>My name is Khai<end>'



### Positional encoding

- Implement positional encoding: [Link](https://kazemnejad.com/blog/transformer_architecture_positional_encoding/)

### MultiHead - Self-attention mechanism

- Implement self-attention with Q, K, V approach

→ Revise on how to implement Masked multi-head attention which prevents the model from attending to subsequent time step from the current time step



# 3. Training model

**Train your model**

- Pretrained model for a few peochs
- Log training/validation loss and compute **perplexity**.
- Save checkpoints and final model.

**Generate text samples:**

- Use your trained model to generate coherent text related to your chosen domain.
- Show 3–5 examples with different prompts.
- Optionally experiment with **temperature, top-k, top-p sampling**


## **4. Evaluation:**

- Report quantitative results (loss, perplexity).
- Qualitative evaluation: human judgment of coherence and relevance.